In [1]:
import numpy as np
import pandas as pd
import os, random, gc
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split

from catboost import CatBoostRegressor
from xgboost import XGBClassifier  # use classifier for binary AUC

TRAIN_PATH = "../input/tabular-playground-series-may-2022/train.csv"
TEST_PATH = "../input/tabular-playground-series-may-2022/test.csv"
SAMPLE_SUBMISSION_PATH = (
    "../input/tabular-playground-series-may-2022/sample_submission.csv"
)
SUBMISSION_PATH = "submission.csv"

ID = "id"
TARGET = "target"

SEED = 2022


def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything()




In [2]:
def reduce_memory_usage(df, verbose=True):
    numerics = ["int8", "int16", "int32", "int64", "float16", "float32", "float64"]
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if (
                    c_min > np.finfo(np.float16).min
                    and c_max < np.finfo(np.float16).max
                ):
                    df[col] = df[col].astype(np.float16)
                elif (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(
            "Mem. usage decreased to {:.2f} Mb ({:.1f}% reduction)".format(
                end_mem, 100 * (start_mem - end_mem) / start_mem
            )
        )
    return df




In [3]:
# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)




In [4]:
# Reduce memory footprint
train = reduce_memory_usage(train)
test = reduce_memory_usage(test)
gc.collect()




Mem. usage decreased to 45.01 Mb (77.7% reduction)
Mem. usage decreased to 5.53 Mb (77.3% reduction)


0

In [5]:
# Prepare features/target; drop column f_27 as in original script
y = train[TARGET]
X = train.drop([ID, TARGET, "f_27"], axis=1)
X_test = test.drop([ID, "f_27"], axis=1)




In [6]:
MODEL_TREE_METHOD = "hist"

# Use XGBClassifier with binary logistic objective suitable for AUC
model = XGBClassifier(
    tree_method=MODEL_TREE_METHOD,
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=SEED,
    n_estimators=300,  # modest number to stay within time limits
    learning_rate=0.1,
    max_depth=6,
)

model.fit(X, y)




XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, random_state=2022, ...)

In [7]:
# Predict probabilities for the positive class and keep full precision
test_pred = model.predict_proba(X_test)[:, 1]

# Ensure predictions are within [0,1] (they already are, but clip as safety)
test_pred = np.clip(test_pred, 0, 1)

sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
sub[TARGET] = test_pred
sub.to_csv(SUBMISSION_PATH, index=False)

# Display a few rows to confirm
sub.head()

,id,target
0,800000,0.861083
1,800001,0.457579
2,800002,0.975887
3,800003,0.456819
4,800004,0.254213
